In [ ]:
import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## Configuration

In [ ]:
# Path to experiment directory
EXPERIMENT_DIR = Path("experiments/2026-04-03_00-00-00")  # <-- update this

# Run labels and descriptions
RUN_INFO = {
    "A": {"label": "GPipe (no interf)", "color": "#2196f3"},
    "B": {"label": "Shisha (no interf)", "color": "#4caf50"},
    "C": {"label": "GPipe (interf)", "color": "#f44336"},
    "D": {"label": "Exhaustive (interf)", "color": "#ff9800"},
    "E": {"label": "Shisha (interf)", "color": "#9c27b0"},
}

In [ ]:
# Load all available runs
runs = {}
for run_id in ["A", "B", "C", "D", "E"]:
    path = EXPERIMENT_DIR / f"run_{run_id}.json"
    if path.exists():
        with open(path) as f:
            runs[run_id] = json.load(f)
        print(f"Loaded run {run_id}: {path.name}")
    else:
        print(f"Missing run {run_id}: {path.name}")

# Load experiment metadata
meta_path = EXPERIMENT_DIR / "experiment_meta.json"
if meta_path.exists():
    with open(meta_path) as f:
        experiment_meta = json.load(f)
    print(f"\nExperiment: {experiment_meta}")

# Collect all model names across runs
all_models = set()
for data in runs.values():
    all_models.update(data.get("results", {}).keys())
all_models = sorted(all_models)
print(f"Models: {all_models}")

## Run Summary

In [ ]:
for run_id, data in sorted(runs.items()):
    info = RUN_INFO.get(run_id, {})
    meta = data.get("meta", {})
    optimizer = meta.get("optimizer", "?")
    print(f"=== Run {run_id}: {info.get('label', '?')} ===")
    print(f"  Optimizer: {optimizer}")
    for model in all_models:
        result = data.get("results", {}).get(model)
        if result is None:
            print(f"  {model}: N/A")
            continue
        batches = result.get("batches", [])
        rps = result.get("requests_per_second", 0)
        timed = [b for b in batches if "timing" in b]
        rebalances = sum(1 for b in batches if b.get("rebalance", {}).get("did_rebalance", False))
        at_optimum = sum(1 for b in batches if b.get("rebalance", {}).get("at_optimum", False))
        if timed:
            times = [b["timing"]["end"] - b["timing"]["start"] for b in timed]
            print(f"  {model}: rps={rps:.2f}, batches={len(timed)}, rebalances={rebalances}, "
                  f"at_optimum={at_optimum}, avg_fwd={np.mean(times):.3f}s")
        else:
            print(f"  {model}: rps={rps:.2f}, no timing data")
    print()

## Throughput Comparison (RPS)

In [ ]:
run_ids = sorted(runs.keys())
n_runs = len(run_ids)
n_models = len(all_models)

fig, ax = plt.subplots(figsize=(max(10, n_models * 2.5), 5))

x = np.arange(n_models)
bar_width = 0.8 / n_runs

for i, run_id in enumerate(run_ids):
    info = RUN_INFO.get(run_id, {})
    rps_values = []
    for model in all_models:
        result = runs[run_id].get("results", {}).get(model)
        rps_values.append(result["requests_per_second"] if result else 0)
    offset = (i - n_runs / 2 + 0.5) * bar_width
    ax.bar(x + offset, rps_values, bar_width,
           label=f"{run_id}: {info.get('label', '?')}",
           color=info.get("color", "gray"), alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(all_models, rotation=15, ha="right")
ax.set_ylabel("Requests per second")
ax.set_title("Overall throughput by run")
ax.legend(fontsize="small")
fig.tight_layout()
plt.show()

## Throughput Impact Table

In [ ]:
# Show RPS and % relative to run A (GPipe baseline)
baseline_id = "A"

header = f"{'Model':<20}"
for run_id in run_ids:
    info = RUN_INFO.get(run_id, {})
    header += f"{run_id + ': ' + info.get('label', '?'):>25}"
print(header)
print("-" * len(header))

for model in all_models:
    bl_result = runs.get(baseline_id, {}).get("results", {}).get(model)
    bl_rps = bl_result["requests_per_second"] if bl_result else 0

    row = f"{model:<20}"
    for run_id in run_ids:
        result = runs[run_id].get("results", {}).get(model)
        if result:
            rps = result["requests_per_second"]
            if bl_rps > 0:
                pct = (rps / bl_rps) * 100
                row += f"{rps:>12.2f} ({pct:>5.0f}%)"
            else:
                row += f"{rps:>12.2f}       "
        else:
            row += f"{'N/A':>25}"
    print(row)

## Batch Times — All Runs

In [ ]:
n_models = len(all_models)
cols = min(n_models, 2)
rows = math.ceil(n_models / cols)

fig, axes = plt.subplots(rows, cols, figsize=(8 * cols, 4 * rows), squeeze=False)

for idx, model in enumerate(all_models):
    ax = axes[idx // cols][idx % cols]

    for run_id in run_ids:
        result = runs[run_id].get("results", {}).get(model)
        if result is None:
            continue
        batches = result.get("batches", [])
        timed = [b for b in batches if "timing" in b]
        elapsed = [b["timing"]["end"] - b["timing"]["start"] for b in timed]
        if not elapsed:
            continue

        info = RUN_INFO.get(run_id, {})
        ax.plot(range(len(elapsed)), elapsed,
                color=info.get("color", "gray"), alpha=0.7,
                label=f"{run_id}: {info.get('label', '?')}",
                marker=".", markersize=1, linewidth=0.8)

    ax.set_title(model)
    ax.set_xlabel("Batch index")
    ax.set_ylabel("Forward time (s)")
    ax.legend(fontsize="x-small", loc="upper right")

for idx in range(n_models, rows * cols):
    axes[idx // cols][idx % cols].set_visible(False)

fig.suptitle("Per-batch forward time — all runs", fontsize=14)
fig.tight_layout()
plt.show()

## Batch Times — No Interference (A vs B)

In [ ]:
no_interf_runs = {k: v for k, v in runs.items() if k in ("A", "B")}
if no_interf_runs:
    fig, axes = plt.subplots(1, len(all_models), figsize=(7 * len(all_models), 4), squeeze=False)

    for idx, model in enumerate(all_models):
        ax = axes[0][idx]
        for run_id, data in sorted(no_interf_runs.items()):
            result = data.get("results", {}).get(model)
            if result is None:
                continue
            timed = [b for b in result.get("batches", []) if "timing" in b]
            elapsed = [b["timing"]["end"] - b["timing"]["start"] for b in timed]
            if not elapsed:
                continue
            info = RUN_INFO.get(run_id, {})
            ax.plot(range(len(elapsed)), elapsed,
                    color=info.get("color", "gray"), alpha=0.8,
                    label=f"{run_id}: {info.get('label', '?')}",
                    marker=".", markersize=2)
        ax.set_title(model)
        ax.set_xlabel("Batch index")
        ax.set_ylabel("Forward time (s)" if idx == 0 else "")
        ax.legend(fontsize="small")

    fig.suptitle("No interference: GPipe vs Shisha", fontsize=14)
    fig.tight_layout()
    plt.show()
else:
    print("Runs A and B not available")

## Batch Times — Interference (C vs D vs E)

In [ ]:
interf_runs = {k: v for k, v in runs.items() if k in ("C", "D", "E")}
if interf_runs:
    fig, axes = plt.subplots(1, len(all_models), figsize=(7 * len(all_models), 4), squeeze=False)

    for idx, model in enumerate(all_models):
        ax = axes[0][idx]
        for run_id, data in sorted(interf_runs.items()):
            result = data.get("results", {}).get(model)
            if result is None:
                continue
            timed = [b for b in result.get("batches", []) if "timing" in b]
            elapsed = [b["timing"]["end"] - b["timing"]["start"] for b in timed]
            if not elapsed:
                continue
            info = RUN_INFO.get(run_id, {})
            ax.plot(range(len(elapsed)), elapsed,
                    color=info.get("color", "gray"), alpha=0.8,
                    label=f"{run_id}: {info.get('label', '?')}",
                    marker=".", markersize=1, linewidth=0.8)

        # Add run A mean as baseline reference
        if "A" in runs:
            a_result = runs["A"].get("results", {}).get(model)
            if a_result:
                a_timed = [b for b in a_result.get("batches", []) if "timing" in b]
                a_times = [b["timing"]["end"] - b["timing"]["start"] for b in a_timed]
                if a_times:
                    ax.axhline(np.mean(a_times), color=RUN_INFO["A"]["color"],
                               linestyle="--", alpha=0.5, label="A: GPipe avg")

        ax.set_title(model)
        ax.set_xlabel("Batch index")
        ax.set_ylabel("Forward time (s)" if idx == 0 else "")
        ax.legend(fontsize="x-small")

    fig.suptitle("Under interference: GPipe vs Exhaustive vs Shisha", fontsize=14)
    fig.tight_layout()
    plt.show()
else:
    print("Runs C, D, E not available")

## Optimizer State Comparison (D vs E)

In [ ]:
de_runs = {k: v for k, v in runs.items() if k in ("D", "E")}
if de_runs:
    for model in all_models:
        fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=False)
        fig.suptitle(f"{model} — Optimizer State", fontsize=14)

        for run_id, data in sorted(de_runs.items()):
            result = data.get("results", {}).get(model)
            if result is None:
                continue
            batches = result.get("batches", [])
            info = RUN_INFO.get(run_id, {})
            color = info.get("color", "gray")
            label = f"{run_id}: {info.get('label', '?')}"

            deep_gamma = [b.get("rebalance", {}).get("deep_gamma") for b in batches]
            sibling_gamma = [b.get("rebalance", {}).get("sibling_gamma") for b in batches]
            best_tp = [b.get("rebalance", {}).get("best_throughput") for b in batches]

            if not any(v is not None for v in deep_gamma):
                continue

            opt_kwargs = data.get("meta", {}).get("optimizer_kwargs", {})
            deep_alpha = opt_kwargs.get("deep_alpha", 5)

            combined = [
                (d or 0) + (s or 0) * deep_alpha
                if d is not None and s is not None else None
                for d, s in zip(deep_gamma, sibling_gamma)
            ]

            xs = range(len(batches))

            # Combined gamma
            axes[0].plot(xs, combined, color=color, alpha=0.7, label=label, linewidth=0.8)

            # Best throughput
            axes[1].plot(xs, best_tp, color=color, alpha=0.7, label=label, linewidth=0.8)

            # Optimum regions
            opt_flags = [b.get("rebalance", {}).get("at_optimum", False) for b in batches]
            axes[2].fill_between(xs, opt_flags, color=color, alpha=0.3, label=label, step="post")

        axes[0].set_ylabel("Combined gamma")
        axes[1].set_ylabel("Best throughput")
        axes[2].set_ylabel("At optimum")
        axes[2].set_xlabel("Batch index")
        axes[2].set_yticks([0, 1])
        axes[2].set_yticklabels(["No", "Yes"])

        for ax in axes:
            ax.legend(fontsize="small", loc="upper right")

        fig.tight_layout()
        plt.show()
else:
    print("Runs D and E not available")

## Forward Time Distribution — Box Plots

In [ ]:
fig, axes = plt.subplots(1, len(all_models), figsize=(5 * len(all_models), 5), squeeze=False)

for idx, model in enumerate(all_models):
    ax = axes[0][idx]
    box_data = []
    box_labels = []
    box_colors = []

    for run_id in run_ids:
        result = runs[run_id].get("results", {}).get(model)
        if result is None:
            continue
        timed = [b for b in result.get("batches", []) if "timing" in b]
        times = [b["timing"]["end"] - b["timing"]["start"] for b in timed]
        if times:
            box_data.append(times)
            info = RUN_INFO.get(run_id, {})
            box_labels.append(f"{run_id}")
            box_colors.append(info.get("color", "gray"))

    if box_data:
        bp = ax.boxplot(box_data, tick_labels=box_labels, patch_artist=True)
        for patch, color in zip(bp["boxes"], box_colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.5)

    ax.set_title(model)
    ax.set_ylabel("Forward time (s)" if idx == 0 else "")
    ax.set_xlabel("Run")

fig.suptitle("Forward time distribution by run", fontsize=14)
fig.tight_layout()
plt.show()

## Rebalance Activity

In [ ]:
print(f"{'Model':<20} {'Run':<8} {'Batches':>10} {'Rebalances':>12} {'At Optimum':>12} {'Rebal %':>10} {'Optimum %':>10}")
print("-" * 82)

for model in all_models:
    for run_id in run_ids:
        result = runs[run_id].get("results", {}).get(model)
        if result is None:
            continue
        batches = result.get("batches", [])
        timed = [b for b in batches if "timing" in b]
        n = len(timed)
        rebalances = sum(1 for b in batches if b.get("rebalance", {}).get("did_rebalance", False))
        at_optimum = sum(1 for b in batches if b.get("rebalance", {}).get("at_optimum", False))
        rebal_pct = (rebalances / n * 100) if n > 0 else 0
        opt_pct = (at_optimum / n * 100) if n > 0 else 0
        info = RUN_INFO.get(run_id, {})
        print(f"{model:<20} {run_id:<8} {n:>10} {rebalances:>12} {at_optimum:>12} {rebal_pct:>9.1f}% {opt_pct:>9.1f}%")